# 精确 f696 账户的开盘与恢复接收层支持

只读封存原生 Qlib 账户路径和已核实的 HOLD 现金上界；不拟合、不改变策略。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R

root=Path.cwd();card=root/"research/stage1_f696_receiver_action_support_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["sources"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and all(v==0 for v in plan["budget"].values())
a=root/src["stage1_card"];b=root/src["hold_bound_card"]
assert sha(a/"summary.json")==src["stage1_summary_sha256"]
assert sha(a/"readback_verification.json")==src["stage1_readback_sha256"]
assert sha(b/"result.json")==src["hold_bound_result_sha256"]
sa=json.loads((a/"summary.json").read_text());ra=json.loads((a/"readback_verification.json").read_text());bound=json.loads((b/"result.json").read_text())
assert sa["nas_evidence"]["sha256"]==src["stage1_nas_sha256"] and sa["nas_evidence"]["size_bytes"]==src["stage1_nas_size_bytes"]
assert ra["archive_sha256"]==src["stage1_nas_sha256"] and ra["status"]=="passed" and ra["all_artifact_hashes_match"]
artifact=a/"qlib_recorder"/src["stage1_experiment_id"]/src["stage1_recorder_id"]/"artifacts/paths/A_active_research_replay_0"
assert sha(artifact)==src["stage1_path_sha256"]
assert sha(root/src["qlib_provider"]/"qlib_store_manifest.json")==src["qlib_provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["qlib_provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None)
with R.uri_context(str(a/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["stage1_experiment_id"],recorder_id=src["stage1_recorder_id"])
 path=rec.load_object("paths/A_active_research_replay_0")
replay=path["replay"].sort_values("trade_date").reset_index(drop=True)
trace=path["trace"].sort_values("trade_date").reset_index(drop=True)
assert len(replay)==len(trace)==bound["sessions"]==944
assert pd.DatetimeIndex(replay.trade_date).is_unique
assert pd.DatetimeIndex(replay.trade_date).equals(pd.DatetimeIndex(trace.trade_date))
calendar=pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2026-10-09",freq="day"))
assert pd.DatetimeIndex(replay.trade_date).isin(calendar).all()
assert replay.execute_open_target.notna().all() and replay.recovery_action.notna().all()
assert replay.execute_open_target.astype(bool).equals(trace.execute_open_target.astype(bool))
assert replay.recovery_action.astype(bool).equals(trace.recovery_action.astype(bool))
open_count=int(replay.execute_open_target.astype(bool).sum());hold_count=len(replay)-open_count
recovery_count=int(replay.recovery_action.astype(bool).sum())
assert open_count==bound["EXECUTE_sessions"]==797 and hold_count==bound["HOLD_sessions"]==147
assert recovery_count==0
annual=[];quarter=[]
for year,g in replay.groupby(pd.DatetimeIndex(replay.trade_date).year,sort=True):
 annual.append({"year":int(year),"sessions":len(g),"partial_endpoint":year in (2023,2026),
                "open_execute":int(g.execute_open_target.astype(bool).sum()),"hold":int((~g.execute_open_target.astype(bool)).sum()),
                "recovery_actions":int(g.recovery_action.astype(bool).sum())})
for (year,q),g in replay.groupby([pd.DatetimeIndex(replay.trade_date).year,pd.DatetimeIndex(replay.trade_date).quarter],sort=True):
 quarter.append({"year":int(year),"quarter":int(q),"sessions":len(g),"open_execute":int(g.execute_open_target.astype(bool).sum()),
                 "hold":int((~g.execute_open_target.astype(bool)).sum()),"recovery_actions":int(g.recovery_action.astype(bool).sum())})
assert len(annual)==4 and len(quarter)==16
window_open=[];window_recovery=[]
for i in range(len(replay)-125):
 g=replay.iloc[i:i+126]
 window_open.append(int(g.execute_open_target.astype(bool).sum()))
 window_recovery.append(int(g.recovery_action.astype(bool).sum()))
assert len(window_open)==len(window_recovery)==bound["rolling126_windows"]==819
assert max(window_recovery)==0
assert bound["original_positive_windows"]==255 and bound["HOLD_only_optimistic_direct_repair_positive_windows"]==450 and bound["required_positive_windows"]==574
result={"status":"read_only_exact_f696_receiver_support_no_candidate",
        "plan_sha256":sha(plan_path),"source_recorder_id":src["stage1_recorder_id"],"sessions":944,"quarters":len(quarter),
        "rolling126_windows":819,"open_execute_sessions":open_count,"hold_sessions":hold_count,"recovery_action_sessions":recovery_count,
        "annual":annual,"quarter":quarter,"rolling126_open_execute_min":min(window_open),"rolling126_open_execute_max":max(window_open),
        "rolling126_recovery_action_max":max(window_recovery),
        "hold_only_frozen_path_bound":{"original_positive_windows":255,"optimistic_negative_hold_cash_erasure_positive_windows":450,"required_positive_windows":574},
        "new_market_requests":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_recorders":0,
        "new_factor_selected":False,"policy_inference":False,"independent_validation":False,
        "limits":["Zero realized recovery actions in baseline does not rule out a separately trained policy that could create new actions",
                  "The HOLD-only erasure freezes the old path and is not an executable own-state account",
                  "All 819 windows overlap; this is support and accounting evidence, not independent validation"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"open":open_count,"hold":hold_count,"recovery":recovery_count,"window_open_min_max":[min(window_open),max(window_open)],"hold_bound":[255,450,574]})


2026/10/24 16:39:10 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[58989:MainThread](2026-10-24 16:39:10,531) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[58989:MainThread](2026-10-24 16:39:10,533) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[58989:MainThread](2026-10-24 16:39:10,533) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_domain_heads_20261018')}


{'open': 797, 'hold': 147, 'recovery': 0, 'window_open_min_max': [87, 119], 'hold_bound': [255, 450, 574]}


## 中文结论

只读封存 f696 A0 0bp Tiered 原生 Qlib 路径、Qlib 交易日历及已核实的 HOLD 现金上界。944 日中 09:46 开盘目标执行 **797** 日、HOLD **147** 日、原 10:32 恢复动作 **0** 日；16 个季度与全部 819 个连续126日窗口保留，任一126日窗口都有87至119个开盘执行日、恢复动作仍为0。0新行情、拟合、账户或阈值搜索。

在**冻结原账户持仓路径**上，乐观抹去全部负的 HOLD 日现金分量，也只将跑赢 SOXX 的滚动窗口从 **255/819** 提到 **450/819**，低于所需 **574/819**。该上界不是可执行账户，更不能证明任何新恢复模型无法触发动作；它说明在当前基线下，仅盯住不执行日或一个从未触发的原恢复分支，缺乏覆盖整体稳定性缺口的证据。

决定 `read_only_prioritize_allocation_and_open_action_receiver_no_candidate`：下一项事前因子应优先说明如何改变 09:29 的原始配置意图，或如何在有大量实际动作日的 09:46 接收层影响资金，并同时经过自身风险上限及完整0bp账户检验。若提出新的10:32恢复模型，需先另证可行动样本、决策时钟及新增动作空间，不能把原0次恢复说成已有有效机会。SOXX目标未达成。
